# Understory API Review



## Nomenclature
Some terms that we use throughout our application and this document that are useful to know

- `Model` - Sometimes also called the `CompanyModel`, this is the "stictched"/"joined" model that we generate using the input of the provided `fileIds`
- `FileModel` - The extracted & parsed data model we build after consuming an uploaded file. This data is used to build the `Model`, and it also contains page, table, and cell `boundingBox`s.

## Reference
- `geometry`
    - `boundingBox` - These are the coordinates encapsulating the given object from the uploaded pdf.
        - The `top`, `left`, `right` and `bottom` are all ratios of the encompassing `page`
        - For example, a `page` will have a `top` and `left` of `0` and a `height` and `width` of `1`
        - A `table` can then have a `top` of `0.10` meaning that it starts 10% down from the top of the `page`, and a `height` of `0.50` meaning that it continues down to `0.60`% of the page (`0.10 + 0.50`)
    - `polygon` - Like `boundingBox` but uses an array of `X/Y` coordinates to draw more complex shapes around the object in question.



# Import and Setup

### Environment Setup

🚨🚨 These steps are required 🚨🚨

1. Run ```cp .env.copy .env```
1. Add your API key to `API_KEY=` in `.env`
1. Also add the domain to `DOMAIN=` for your API, eg `abc-api.understorytech.com`
1. Install dependencies `pip3 install -r requirements.txt`

In [ ]:
import os, re, requests, json
from dotenv import load_dotenv

# Load environment variables
load_dotenv("./.env")

# Setup API configuration
headers = {
    'x-api-key': os.getenv('API_KEY')
}

domain = os.getenv('DOMAIN')
BASE_URL = f"https://{domain}/companies"

print(f"Using BASE_URL: {BASE_URL}")

When we return `4xx` or `5xx` errors, we often also include a helpful message on why the error occured in the message body.
To make this easier to see if something goes wrong with our API usage, we're setting up a utility function to print that message body for us.

In [4]:
def raise_for_status_with_body(response):
    """
    Custom version of raise_for_status that includes response body in error message.
    Args:
        response: requests.Response object
    Raises:
        HTTPError: with detailed error message including response body
    """
    try:
        response.raise_for_status()
    except requests.exceptions.HTTPError as e:
        print(f"HTTP Status Code: {response.status_code}")
        try:
            error_body = response.json()
            print("Error Response Body:", error_body)
        except ValueError:
            print("Error Response Body:", response.text)
        raise e

# Companies

## Get Companies

In [ ]:
res = requests.get(BASE_URL, headers=headers)
raise_for_status_with_body(res)
print(res.json())

## Create a Company

In [ ]:
payload = {"companyName": "Delete me after 1 hour", "deleteAfterHours": 1}
res = requests.post(BASE_URL, headers=headers, json=payload)
raise_for_status_with_body(res)

# Extract the newly created company_id from the response
response_data = res.json()
print(response_data)
company_id = response_data["companyId"]
print(f"Created company with ID: {company_id}")

web_url = re.sub(r'-api(?=\.)', '', domain)
print("🚨 Go to this URL if you want to view your created company in our UI")
print(f"https://{web_url}/company/{company_id}")

## Get Company Files

In [ ]:
res = requests.get(f"{BASE_URL}/{company_id}/files", headers=headers)
raise_for_status_with_body(res)
print(res.json())

# Files

## Create New File

In [9]:
# maintain a list of the created file ids, so that we can provide them when generating a model later
created_file_ids = []

In [ ]:
payload = {
    "filename": "pfizer-encrypted-7.pdf",
    # I picked these pages from our example PDF because they have tables on them
    "pageIngest": [[20,25]]
}

res = requests.post(
    f"{BASE_URL}/{company_id}/files", 
    headers=headers, 
    json=payload
)
raise_for_status_with_body(res)

response_data = res.json()

file_id = response_data["fileId"]
# keep track of the file id so we can use it in model gen later
created_file_ids.append(file_id)
print(f"Created file with ID: {file_id}")

# Get the signed upload url so that we can upload the file directly to s3. 
# NOTE: This signed url has a 900 second expiration.
upload_url = response_data["uploadUrl"]
print(f"Upload URL: {upload_url}")

# We're going to upload our sample PDF file here for testing
with open("pdfs/pfizer-encrypted.pdf", 'rb') as file:
    upload_response = requests.put(upload_url, data=file)
    raise_for_status_with_body(upload_response)
    print("File uploaded successfully!")

## Important Context on Files
Now that the file has been uploaded, Understory will do a bunch of processing to extract data from that file. This is different then what we refer to as the model generation process, as it is done on individual files independent of other files. In order for files to be used in the model generation process, they must finish processing (`status=succeeded`). As a convenience, we've made it easy to queue generating a model so you don't have to wait for these file's to finish processing. Immediately after updating your files, you can send a request to generate a model with the `fileId`s you got, and it will automatically wait until those files have finished processing before generating your model for you.

We'll create some better documentation on this soon, but at present these are the current possible statuses for a file:

```
[
  "created",
  "uploaded",
  "pending", // used when re-processing
  "empty", // file is empty
  "importing_started",
  "importing_completed",
  "textract_started",
  "textract_completed",
  "textract_disabled",
  "parsing_started",
  "parsing_completed",
  "failed"
]
```

# Models

## Create a Model

In [ ]:
print(f"Creating model with file ids: {created_file_ids}")

payload = {
    "includedFileIds": created_file_ids  # Using the list we've been maintaining
}

res = requests.post(
    f"{BASE_URL}/{company_id}/models",
    headers=headers,
    json=payload
)
raise_for_status_with_body(res)
print(res.json())

model_id = res.json()["modelId"]
print(f"Model created with ID: {model_id}")

Now that our model is being created, we need to wait for it to finish to get the generated result. This process can take anywhere from 2-40 minutes based on how much input data (the files, pages, & tables) were given. This can take significant time because this is the process where we do all the 'stitching' (ie: the process of combining all the tables and timeseries across different files) as well as a bunch of other logic in order to derive a clean model.

For now, the best way to check on the status is to simply long-poll the `GET companies/{companyId}/models/{modelId}` to see the status. In the future, we'll provide a callback URL that will send you a notification once it's done.

## Get a Model

In [ ]:
res = requests.get(
    f"{BASE_URL}/{company_id}/models/{model_id}",
    headers=headers
)
raise_for_status_with_body(res)
model_json = res.json()
print(model_json)

if model_json["status"] == "completed":
    print("Model completed! We can download it now")


### Important context on Models

Like files, models also have statuses. As aforementioned, we'll provide better documentation on this soon. These are the current possible statuses for a model:

```
[
  "created",
  "preparing",
  "running",
  "joining",
  "organizing",
  "summarizing",
  "printing",
  "completed",
  "failed",
] 
```

## Get Models

In [ ]:
from IPython.display import JSON
res = requests.get(
    f"{BASE_URL}/{company_id}/models",
    headers=headers
)
raise_for_status_with_body(res)
models_json = res.json()


for model in models_json["items"]:
   print(f"{model['modelId']} - {model['status']}")

## Download a Model

Once the model is ready, we can now download it by getting a signed GET url which we can then use to directly download. Note that if the model isn't ready yet, it will return a `400` error.

In [ ]:
model_id = "1740022601351" # replace with the modelId you want to download
res = requests.post(
    f"{BASE_URL}/{company_id}/models/{model_id}/download",
    headers=headers
)
raise_for_status_with_body(res)

model_download_res = res.json()

# NOTE: there's a bug right now where the `modelType` of this response will say "excel", but this is wrong.
print(model_download_res)

download_url = model_download_res["downloadUrl"]
download_response = requests.get(download_url)
raise_for_status_with_body(download_response)

# Parse the JSON content
model_data = download_response.json()

# Save the JSON to a file
with open("model_output.json", "w") as f:
    json.dump(model_data, f, indent=2)
print("Model saved to model_output.json")

# Practical Examples

## Getting Bounding Boxes for Source Cells

Now that we have the generated model, we can inspect the cells that make up the stitched result.

This example picks a random cell from the model and show how to get the bounding box of the _original_ table that the cell came from. These are seperate endpoints because the `model` is a stictched result that is the combination of multiple files. The `fileModel` endpoint is
just the extracted & parsed data of a single file, which is used as one of the inputs to generate
the model.

### Getting an arbitrary cell

The below code loads an arbitrary cell from the `model`, which we'll use in our example.
This could be a cell that the user has selected in some GUI, for example.

In [ ]:
import json

with open('model_output.json', 'r') as f:
    # picking some arbitray cell
    # NOTE: this may not work for you because of slight changes in the keys. Inspect your
    # 'model_output.json'
    arbitrary_cell = json.load(f)["tableGroups"][0]["tables"][0]["sections"][0]["data"][0]["cells"][0]


cell_coordinates = arbitrary_cell["sourceCoordinates"]
print(f"Cell coordinates: {cell_coordinates}")

file_id = cell_coordinates["fileId"]

### Download a FileModel

To get the extracted properties of individual files, you can download the `FileModel` for a given file from our API. This will provide the necessary `Geometry` data for all of the tables for a given file in order to visualize the locations of cells that were used in the `Model`.

In your application, you'll want to cache/persist these for each file after they are done
uploading to avoid constantly downloading the `FileModel`.

In [ ]:
# Get a signed URL for the FileModel for the ID we selected in the previous step
res = requests.post(
    f"{BASE_URL}/{company_id}/files/{file_id}/download?type=json",
    headers=headers
)
raise_for_status_with_body(res)
file_model_download_res = res.json()

# Extract the downloadUrl from the resposne and download
download_url = file_model_download_res["downloadUrl"]
download_response = requests.get(download_url)
raise_for_status_with_body(download_response)

file_model_data = download_response.json()

# Save the JSON to a file for reference
with open("file_model_output.json", "w") as f:
    if file_model_data["version"] != "0.6.1":
        # If you're getting this error, it means the file model needs to be updated in our system.
        # In order to do this, you can invoke a new company model generation which will automatically
        # update the file models for you.
        raise Exception("This example is only compatible with version 0.6.1 of the API (current version: " + file_model_data["version"] + ")")
    json.dump(file_model_data, f, indent=2)
print("Model saved to file_model_output.json")


### Get the `table` coordinates for our cell in question

In [ ]:
if not file_model_data or not "pages" in file_model_data:
    raise Exception("No pages found in file model")

if not cell_coordinates or not "tableId" in cell_coordinates:
    raise Exception("No tableId found in cell coordinates")

# Get the table that contains our cell

page = next((p for p in file_model_data["pages"] if p["id"] == cell_coordinates["pageId"]), None)
if not page:
    raise Exception(f"Couldn't find page with id {cell_coordinates['pageId']}")

print("Page coordinates: ", page["geometry"]["boundingBox"])

table = next((t for t in page["tables"] if t["id"] == cell_coordinates["tableId"]), None)
if not table:
    raise Exception(f"Couldn't find table with id {cell_coordinates['tableId']}")

# Map the table parts. Parts are usually a single table, but sometimes we detect that
# a table actually spans multiple tables (usually because of page breaks). That's
# why parts is an array.
table_coordinates = [part['geometry']['boundingBox'] for part in table['parts']]
print("Table coordinates: ", table_coordinates)



### Getting the page dimensions

The coordinates retrieved above are just relative (ratios) to the dimensions of the actual pdf. You can get the page dimensions at a system level by doing somehting like this:

In [ ]:
from PyPDF2 import PdfReader

# Open the PDF
reader = PdfReader("AcmeStores 2021 FQ3PR.pdf")

# Get the first page (pages are 0-indexed)
page = reader.pages[0]

# Get the page dimensions (in points, where 1 point = 1/72 inches)
height = page.mediabox.height
width = page.mediabox.width

print(f"Page dimensions: {height} x {width}")

# If you want it in inches
height_inches = float(height) / 72
width_inches = float(width) / 72

# If you want it in mm
height_mm = height_inches * 25.4
width_mm = width_inches * 25.4

